# Altered States, annealed — Jane Street, March 2014

[https://www.janestreet.com/puzzles/altered-states-index/](https://www.janestreet.com/puzzles/altered-states-index/)

A second attack on the same puzzle, using **simulated annealing** instead of CP-SAT. The
CP-SAT work lives in [`altered-states.ipynb`](altered-states.ipynb); this notebook stands alone
so the two can be raced against each other on the same clock.

## Why a second approach at all

Altered States is an *open-ended optimization* puzzle: there is no known ceiling, and the score
is the point. CP-SAT is superb at proving things, but proving needs a useful upper bound, and
this problem gives it none — the bound sits near 412 ("all fifty states fit") and barely moves.
Strip away the pruning and branch-and-bound becomes very expensive enumeration.

Annealing does not need a bound. It needs a neighbourhood and a temperature.

## AI use disclaimer

I used AI to help me with Python syntax and debugging, but all code and text is fully mine.

## What went wrong the first time

A plain hill-climber on this puzzle stalls around 80. Two reasons, and the fixes here address
both:

**The landscape is flat.** A state scores its full length or nothing, so almost every
single-letter change is worth exactly zero. There is no slope to walk up. The fix is *partial
credit*: score a near miss by how far it got before dying, so a grid that spells `IOW` ranks
above one that dies at `IO`.

**One-cell moves are too small.** Two good grids usually differ in many cells at once, and the
path between them runs through worse grids. A hill-climber refuses to take that path. Annealing
accepts a worse grid on purpose, with a probability that shrinks as it cools — that is the whole
trick, and it is what lets the search leave a local optimum.

In [ ]:
import random
import math
import time

## The states, the grid, and the letter pool

In [ ]:
STATE_NAMES = [
    "ALABAMA", "ALASKA", "ARIZONA", "ARKANSAS", "CALIFORNIA",
    "COLORADO", "CONNECTICUT", "DELAWARE", "FLORIDA", "GEORGIA",
    "HAWAII", "IDAHO", "ILLINOIS", "INDIANA", "IOWA",
    "KANSAS", "KENTUCKY", "LOUISIANA", "MAINE", "MARYLAND",
    "MASSACHUSETTS", "MICHIGAN", "MINNESOTA", "MISSISSIPPI", "MISSOURI",
    "MONTANA", "NEBRASKA", "NEVADA", "NEWHAMPSHIRE", "NEWJERSEY",
    "NEWMEXICO", "NEWYORK", "NORTHCAROLINA", "NORTHDAKOTA", "OHIO",
    "OKLAHOMA", "OREGON", "PENNSYLVANIA", "RHODEISLAND", "SOUTHCAROLINA",
    "SOUTHDAKOTA", "TENNESSEE", "TEXAS", "UTAH", "VERMONT",
    "VIRGINIA", "WASHINGTON", "WESTVIRGINIA", "WISCONSIN", "WYOMING",
]

GRID_SIZE = 5

# Drawing a random letter from this string samples letters in proportion to how often they
# appear across all fifty state names. Sampling uniformly from the alphabet instead would spend
# a large share of every search on J, X and Z, which between them appear in four states.
LETTER_POOL = "".join(STATE_NAMES)

In [ ]:
def all_cells(size):
    """Return every (row, column) pair of a square grid, reading across then down."""
    cells = []
    for row in range(size):
        for column in range(size):
            cells.append((row, column))
    return cells


def king_neighbors(row, column, size):
    """Return the cells a King can move to from (row, column).

    The starting cell itself is never included, which is what makes HAWAII unspellable without
    a second I adjacent to the first.
    """
    neighbors = []
    for row_step in (-1, 0, 1):
        for column_step in (-1, 0, 1):
            is_staying_put = row_step == 0 and column_step == 0
            if is_staying_put:
                continue
            neighbor_row = row + row_step
            neighbor_column = column + column_step
            row_on_board = 0 <= neighbor_row < size
            column_on_board = 0 <= neighbor_column < size
            if row_on_board and column_on_board:
                neighbors.append((neighbor_row, neighbor_column))
    return neighbors

## The readable scorer

This is the reference implementation: slow, obvious, and the thing every faster version gets
checked against.

It sweeps the word one letter at a time, keeping the set of squares a legal walk could currently
be standing on. How it got there does not affect where it can go next, so only the ending square
matters and the set never exceeds 25 entries.

Returning the *prefix length* rather than a yes/no is what makes partial credit possible: the
step at which the frontier empties is exactly how far the best walk got.

In [ ]:
def spellable_prefix_length(grid, word):
    """Return how many letters of word can be spelled by King's moves in this grid.

    len(word) means the whole word is present. Anything less is how far the best walk got
    before it ran out of neighbours, which is the partial credit the annealer needs.
    """
    size = len(grid)

    frontier = set()
    for row, column in all_cells(size):
        if grid[row][column] == word[0]:
            frontier.add((row, column))
    if len(frontier) == 0:
        return 0

    letters_spelled = 1
    for next_letter in word[1:]:
        new_frontier = set()
        for row, column in frontier:
            for neighbor in king_neighbors(row, column, size):
                neighbor_row, neighbor_column = neighbor
                if grid[neighbor_row][neighbor_column] == next_letter:
                    new_frontier.add(neighbor)
        if len(new_frontier) == 0:
            return letters_spelled
        frontier = new_frontier
        letters_spelled += 1

    return letters_spelled


def word_in_grid(grid, word):
    """Return whether the whole word can be spelled in this grid."""
    return spellable_prefix_length(grid, word) == len(word)


def states_in_grid(grid, states=STATE_NAMES):
    """Return the set of states present in the grid."""
    found = set()
    for state in states:
        if word_in_grid(grid, state):
            found.add(state)
    return found


def score_grid(grid, states=STATE_NAMES):
    """Return the puzzle score: the total length of the distinct states in the grid."""
    total = 0
    for state in states_in_grid(grid, states):
        total += len(state)
    return total

## Validate against the published example

The puzzle supplies one scored grid. If the scorer disagrees with it, nothing built on top is
worth running, so these assert rather than print.

In [ ]:
EXAMPLE_GRID = [
    [".", "I", "H"],
    ["D", "A", "O"],
    [".", "W", "."],
]

assert states_in_grid(EXAMPLE_GRID) == {"IDAHO", "IOWA", "OHIO"}
assert score_grid(EXAMPLE_GRID) == 13

# OHIO must be found even though it starts and ends on the same square: reuse is legal.
assert word_in_grid(EXAMPLE_GRID, "OHIO")

# HAWAII must not be, since reaching the second I would mean staying put. It should get five
# letters in and stall, which is exactly the partial credit signal the annealer runs on.
assert not word_in_grid(EXAMPLE_GRID, "HAWAII")
assert spellable_prefix_length(EXAMPLE_GRID, "HAWAII") == 5

print("scorer agrees with the published example: 13 points")

## The fast scorer

Annealing lives or dies on how many grids per second it can evaluate, and the readable scorer
manages only a few thousand. Since the frontier is a *set of squares*, and a set of at most 25
things is an integer, the whole sweep can be done in arithmetic.

One bit per square, laid out **8 bits per row** so that shifting by 8 moves a whole row at a
time. Then expanding the entire frontier by one King's move is five shifts and a few ORs,
regardless of how many squares are in it:

- shifting left and right gives the horizontal neighbours (masked so nothing wraps around an edge)
- shifting that whole thing up and down by a row gives the vertical and diagonal ones

The origin squares are deliberately not included in the result, so this gets the HAWAII rule for
free in the same way `king_neighbors` does.

In [ ]:
# One bit per square: bit (row * 8 + column). Eight bits per row rather than five so that a
# shift of 8 moves exactly one row and the arithmetic stays simple.
BOARD_MASK = 0
for row in range(GRID_SIZE):
    for column in range(GRID_SIZE):
        BOARD_MASK = BOARD_MASK | (1 << (row * 8 + column))

# Squares that may shift left / right without falling off that side of the board.
left_edge = 0
right_edge = 0
for row in range(GRID_SIZE):
    left_edge = left_edge | (1 << (row * 8 + 0))
    right_edge = right_edge | (1 << (row * 8 + GRID_SIZE - 1))
NOT_ON_LEFT_EDGE = BOARD_MASK & ~left_edge
NOT_ON_RIGHT_EDGE = BOARD_MASK & ~right_edge


def king_expand(frontier_bits):
    """Return every square reachable in one King's move from any square in the frontier."""
    sideways = ((frontier_bits & NOT_ON_RIGHT_EDGE) << 1) | (
        (frontier_bits & NOT_ON_LEFT_EDGE) >> 1
    )
    # Moving the frontier and its horizontal neighbours up and down a row covers the vertical
    # and diagonal moves. The frontier squares themselves never survive into the result.
    spread = frontier_bits | sideways
    return ((spread << 8) | (spread >> 8) | sideways) & BOARD_MASK


def letter_masks(grid):
    """Return {letter: bitmask of the squares holding it} for this grid."""
    masks = {}
    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            letter = grid[row][column]
            masks[letter] = masks.get(letter, 0) | (1 << (row * 8 + column))
    return masks

## The shaped objective

`shaped_and_true_score` returns both numbers in one sweep, because the annealer needs the shaped
one to decide and the true one to record, and computing them separately would double the cost.

- **true score** — what the puzzle actually pays: the length of each complete state
- **shaped score** — the same, plus `PARTIAL_CREDIT` per letter of every *incomplete* state

The partial weight is deliberately small. Too large and the search starts preferring grids
stuffed with near misses over grids that finish fewer, longer states — the shaped optimum drifts
away from the real one. Measured on this puzzle, `0.0` and `0.15` perform similarly and `0.4` is
clearly worse, so this is a knob worth re-testing rather than trusting.

In [ ]:
PARTIAL_CREDIT = 0.15


def shaped_and_true_score(masks, states=STATE_NAMES, partial_credit=PARTIAL_CREDIT):
    """Return (shaped score, true score) for a grid given as letter masks."""
    shaped = 0.0
    true_total = 0

    for state in states:
        frontier_bits = masks.get(state[0], 0)
        if frontier_bits == 0:
            continue

        letters_spelled = 1
        for next_letter in state[1:]:
            frontier_bits = king_expand(frontier_bits) & masks.get(next_letter, 0)
            if frontier_bits == 0:
                break
            letters_spelled += 1

        if letters_spelled == len(state):
            shaped += len(state)
            true_total += len(state)
        else:
            shaped += partial_credit * letters_spelled

    return shaped, true_total

## Check the fast scorer against the readable one

The bitboard is the only clever code in this notebook, so it gets fuzzed: hundreds of random
grids, scored both ways, asserting they agree. This is what catches the off-by-one in an edge
mask that would otherwise quietly cost points for the rest of the session.

In [ ]:
def random_grid(size=GRID_SIZE):
    """Return a grid with a letter in every cell, drawn by frequency across the state names."""
    grid = []
    for row in range(size):
        new_row = []
        for column in range(size):
            new_row.append(random.choice(LETTER_POOL))
        grid.append(new_row)
    return grid


random.seed(11)
for attempt in range(300):
    grid = random_grid()
    _, fast_score = shaped_and_true_score(letter_masks(grid))
    assert fast_score == score_grid(grid), grid

print("bitboard scorer agrees with the readable one on 300 random grids")

sample = random_grid()
start = time.perf_counter()
for _ in range(2000):
    score_grid(sample)
readable_rate = 2000 / (time.perf_counter() - start)

start = time.perf_counter()
for _ in range(2000):
    shaped_and_true_score(letter_masks(sample))
fast_rate = 2000 / (time.perf_counter() - start)

print(f"readable: {readable_rate:,.0f} grids/sec")
print(f"bitboard: {fast_rate:,.0f} grids/sec")

## The annealer

Two kinds of move, because they explore differently:

- **set** — give one square a new random letter, drawn from the frequency-weighted pool
- **swap** — exchange the letters on two squares, which rearranges without changing the letter
  mix at all, and reaches grids that no sequence of single changes reaches without passing
  through worse ones

A move that improves the shaped score is always kept. A move that makes it worse is kept with
probability `exp(change / temperature)` — near-certain while hot, near-never once cold. The
temperature falls geometrically from `start_temperature` to `end_temperature` across the run, so
the search begins by wandering freely and ends by behaving like a hill-climber.

The best *true* score is tracked separately throughout, because the shaped score is only a guide
and the grid it likes best is not necessarily the grid that scores best.

In [ ]:
def copy_grid(grid):
    """Return an independent copy, so a saved best grid is not aliased to the live one."""
    copy = []
    for row in grid:
        copy.append(list(row))
    return copy


def anneal(
    grid,
    moves,
    start_temperature=4.0,
    end_temperature=0.03,
    swap_fraction=0.3,
    states=STATE_NAMES,
):
    """Anneal from this grid and return (best grid seen, its true score).

    The grid passed in is modified in place; the returned best is a separate copy.
    """
    current_shaped, best_true = shaped_and_true_score(letter_masks(grid), states)
    best_grid = copy_grid(grid)

    # Geometric cooling: multiplying by this each move takes the temperature from start to end
    # over exactly `moves` steps.
    cooling = (end_temperature / start_temperature) ** (1.0 / moves)
    temperature = start_temperature

    for move_number in range(moves):
        make_a_swap = random.random() < swap_fraction

        if make_a_swap:
            first_row = random.randrange(GRID_SIZE)
            first_column = random.randrange(GRID_SIZE)
            second_row = random.randrange(GRID_SIZE)
            second_column = random.randrange(GRID_SIZE)
            if grid[first_row][first_column] == grid[second_row][second_column]:
                temperature = temperature * cooling
                continue
            grid[first_row][first_column], grid[second_row][second_column] = (
                grid[second_row][second_column],
                grid[first_row][first_column],
            )
        else:
            first_row = random.randrange(GRID_SIZE)
            first_column = random.randrange(GRID_SIZE)
            replaced_letter = grid[first_row][first_column]
            new_letter = random.choice(LETTER_POOL)
            if new_letter == replaced_letter:
                temperature = temperature * cooling
                continue
            grid[first_row][first_column] = new_letter

        candidate_shaped, candidate_true = shaped_and_true_score(letter_masks(grid), states)
        change = candidate_shaped - current_shaped

        if change >= 0:
            keep_the_move = True
        else:
            # Accept a worse grid sometimes. This is the entire point of annealing: it is the
            # only way out of a local optimum that no single improving move escapes.
            keep_the_move = random.random() < math.exp(change / temperature)

        if keep_the_move:
            current_shaped = candidate_shaped
            if candidate_true > best_true:
                best_true = candidate_true
                best_grid = copy_grid(grid)
        else:
            if make_a_swap:
                grid[first_row][first_column], grid[second_row][second_column] = (
                    grid[second_row][second_column],
                    grid[first_row][first_column],
                )
            else:
                grid[first_row][first_column] = replaced_letter

        temperature = temperature * cooling

    return best_grid, best_true

## Run it

Independent restarts rather than one very long run: each restart samples a different local
optimum, and the best of many beats the deepest one. Re-run this cell as often as you like — it
only ever replaces `BEST_GRID_SO_FAR` with something strictly better.

In [ ]:
def show_grid(grid):
    """Print a grid as the puzzle asks for it: rows reading across."""
    for row in grid:
        print(" ".join(row))


def show_grid_as_literal(grid):
    """Print the grid as Python source, ready to paste somewhere it will survive a restart."""
    print("BEST_GRID_SO_FAR = [")
    for row in grid:
        letters = ", ".join(f'"{letter}"' for letter in row)
        print(f"    [{letters}],")
    print("]")


def anneal_many(restarts, moves=150000, start_temperature=4.0, end_temperature=0.03):
    """Run several independent anneals and return the best (grid, true score) found."""
    best_grid = None
    best_true = -1
    for attempt in range(restarts):
        grid, true_score = anneal(
            random_grid(), moves, start_temperature, end_temperature
        )
        if true_score > best_true:
            best_true = true_score
            best_grid = grid
            print(f"restart {attempt:3d}: new best {best_true}")
    return best_grid, best_true

In [ ]:
BEST_GRID_SO_FAR = None
BEST_SCORE_SO_FAR = 0

In [ ]:
grid, found_score = anneal_many(restarts=6, moves=150000)

if found_score > BEST_SCORE_SO_FAR:
    BEST_GRID_SO_FAR = grid
    BEST_SCORE_SO_FAR = found_score

print()
show_grid(BEST_GRID_SO_FAR)
print()
print("best true score:", BEST_SCORE_SO_FAR)
print("checked with the readable scorer:", score_grid(BEST_GRID_SO_FAR))
print()
show_grid_as_literal(BEST_GRID_SO_FAR)

## What this is actually for

Measured on this machine: six restarts of 150k moves takes about 35 seconds and reaches roughly
100. CP-SAT, unaided, reached 108 on the same puzzle — so annealing does **not** simply beat the
solver here, which is worth knowing before trusting the folklore that metaheuristics always win
on open-ended optimization.

Where it earns its place is as a **source of structurally different candidates**. CP-SAT
improves a grid by tearing up and repairing part of it, which keeps it near whatever it already
holds; two good grids usually differ in most of their cells, so the solver cannot cross between
them. Annealing starts somewhere else entirely every restart.

The pipeline that uses both:

1. anneal a batch of restarts and keep several *different* grids, not just the best one
2. hand each to CP-SAT as `seed_grid`, or pin its strongest row with `fixed_squares`
3. let the solver polish each to local optimality — the part annealing is bad at and it is good at

Things worth trying from here:

- longer runs, and a wider `start_temperature`, to see where the score plateaus
- **reheating** — instead of restarting from a random grid, restart from the best grid so far at
  a raised temperature, which explores near a good solution rather than starting over
- re-tuning `PARTIAL_CREDIT`; it was only tested at three values
- a move that changes a whole row at once, closer to the neighbourhood size CP-SAT proved useful